# 01 · Fase 1: busca léxica no OpenSearch


In [35]:
from pathlib import Path
import sys, json, shutil
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ri_core.py").is_file()), None)
assert ROOT is not None, "Extraia o pacote e abra o notebook a partir da sua pasta."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ri_core import load_data, load_qrels, load_run, evaluate, write_json, write_csv
from ri_workflow import *
print("Raiz do pacote:", ROOT)

Raiz do pacote: C:\Users\denis\OneDrive\Área de Trabalho\lab_ri_opensearch_Fase_01_Equipe_02\lab_ri_opensearch_Fase_01_Equipe_02


Pré-requisito: notebook 00 homologado, OpenSearch 3.9.0 em execução.
 Entrada: corpus e consultas públicos. Saídas: índice, ranking e métricas de validação.
 Para repetir um experimento use novo `EXPERIMENT`, pois índices e pastas existentes não são apagados.

In [40]:
from os_retrieval import run_experiment, OpenSearchHTTP
TEAM = "equipe_02"
EXPERIMENT = "lexico_v4"
CONFIG = json.loads((ROOT / "starter/config.json").read_text(encoding="utf-8"))
CONFIG.update(team_id=TEAM, mode="lexical", analyzer="portuguese", bm25_k1=0.9, bm25_b=.75)
DATA = ROOT / "data/dev"
OUT = ROOT / "resultados" / f"{TEAM}_{EXPERIMENT}"
INDEX = f"ri-{TEAM}-{EXPERIMENT}"
OpenSearchHTTP().check()

{'name': 'ad59490395b8',
 'cluster_name': 'docker-cluster',
 'cluster_uuid': 'X5L_P2PQT_ufNUQlI0igDA',
 'version': {'distribution': 'opensearch',
  'number': '3.9.0',
  'build_type': 'tar',
  'build_hash': '4ee42a94e87f66fbf1e62a9871b1b87f91e02472',
  'build_date': '2026-09-23T01:01:31.883915140Z',
  'build_snapshot': False,
  'lucene_version': '10.5.1',
  'minimum_wire_compatibility_version': '2.19.0',
  'minimum_index_compatibility_version': '2.0.0'},
 'tagline': 'The OpenSearch Project: https://opensearch.org/'}

In [37]:
meta = run_experiment(DATA, OUT, CONFIG, INDEX)
print(meta)

ValueError: Destino deve estar vazio: C:\Users\denis\OneDrive\Área de Trabalho\lab_ri_opensearch_Fase_01_Equipe_02\lab_ri_opensearch_Fase_01_Equipe_02\resultados\equipe_02_lexico_v4

In [38]:
docs, queries = load_data(DATA)
_, ranking = load_run(OUT / "run.csv", docs, queries)
qrels = load_qrels(DATA / "qrels.csv",docs,queries)
per_query, summary = evaluate(ranking,queries,qrels)
write_json(OUT / "metrics.json",summary)
write_csv(OUT / "per_query.csv",list(per_query[0]),per_query)
import csv
COMPARISON = ROOT / "resultados/comparison_lab4.csv"
row = dict(experiment=EXPERIMENT,analyzer=CONFIG["analyzer"],bm25_k1=CONFIG["bm25_k1"],bm25_b=CONFIG["bm25_b"],**{k:summary[k] for k in ["ndcg@10","recall@100","mrr@10"]})
with COMPARISON.open("a",encoding="utf-8",newline="") as f:
    writer=csv.DictWriter(f,fieldnames=list(row))
    if f.tell()==0: writer.writeheader()
    writer.writerow(row)
print(summary, "Comparação:", COMPARISON)

{'ndcg@10': 0.8531730946825415, 'mrr@10': 0.8380952380952381, 'recall@100': 0.9666666666666667, 'judged@10': 0.09000000000000001, 'queries': 30, 'missing_queries': 0} Comparação: C:\Users\denis\OneDrive\Área de Trabalho\lab_ri_opensearch_Fase_01_Equipe_02\lab_ri_opensearch_Fase_01_Equipe_02\resultados\comparison_lab4.csv


Execute ao menos duas configurações, mudando `EXPERIMENT` e o analyzer ou parâmetros BM25. Anote métricas e escolha a versão final no README_EQUIPE.md.
 O `run.py` recebe `--data`, `--out` e `--index`. A pasta gerada abaixo é apenas o início da entrega: acrescente comparison.csv, README_EQUIPE.md e dependências próprias.
 A organização congelará o código antes de executar o teste agregado.

In [41]:
DELIVERY = ROOT / "entregas" / f"{TEAM}_phase1_solution"
fresh(DELIVERY)
shutil.copy2(ROOT / "starter/run.py", DELIVERY / "run.py")
for module in ["os_retrieval.py", "ri_core.py", "ri_workflow.py"]:
    shutil.copy2(ROOT / module, DELIVERY / module)
write_json(DELIVERY / "config.json", CONFIG)
shutil.copy2(ROOT / "starter/README_EQUIPE_LAB4.md", DELIVERY / "README_EQUIPE.md")
shutil.copy2(ROOT / "starter/requirements-extra.txt", DELIVERY / "requirements-extra.txt")
(DELIVERY / "resultados_dev").mkdir()
shutil.copy2(COMPARISON, DELIVERY / "resultados_dev/comparison.csv")
print("Pasta inicial da entrega; complete os arquivos exigidos no README:", DELIVERY)

Pasta inicial da entrega; complete os arquivos exigidos no README: C:\Users\denis\OneDrive\Área de Trabalho\lab_ri_opensearch_Fase_01_Equipe_02\lab_ri_opensearch_Fase_01_Equipe_02\entregas\equipe_02_phase1_solution
